In [0]:
from pyspark.sql.functions import to_date, col, lit

In [0]:
selected_trips = spark.read.table("samples.nyctaxi.trips").select(to_date(col("tpep_pickup_datetime")).alias("pickup_date"), to_date(col("tpep_dropoff_datetime")).alias("dropoff_date"), "trip_distance", "fare_amount", "pickup_zip", "dropoff_zip")

In [0]:
filtered_trips = selected_trips.filter((col("pickup_zip") != col("dropoff_zip")) & (col("pickup_zip") != 0) & (col("dropoff_zip") != 0))

In [0]:
# Add a ride_bonus_amount column: 4% of the fare_amount as a bonus surcharge
filtered_trips_new = filtered_trips.withColumn("ride_bonus_amount", col("fare_amount") * 0.04)

# Group by pickup and dropoff zip codes and aggregate fare, distance, and bonus
aggregated_trips = (
    filtered_trips_new.groupBy("pickup_zip", "dropoff_zip")
    .agg(
        {"fare_amount": "avg", "trip_distance": "avg", "ride_bonus_amount": "avg"}
    )
    .withColumnRenamed("avg(fare_amount)", "avg_fare_amount")
    .withColumnRenamed("avg(trip_distance)", "avg_trip_distance")
    .withColumnRenamed("avg(ride_bonus_amount)", "avg_ride_bonus_amount")
)

In [0]:
grouped_trips = filtered_trips_new.groupBy("pickup_zip", "dropoff_zip").agg({"fare_amount": "avg", "trip_distance": "avg"})

In [0]:
ordered_trips = grouped_trips.orderBy("avg(fare_amount)", "avg(trip_distance)", ascending=False)

In [0]:

ordered_trips.show(10)

In [0]:
total_rows=ordered_trips.count()
print("Total rows: ", total_rows)

In [0]:
local_data = ordered_trips.collect()

for row in local_data:
    print(row['pickup_zip'])

In [0]:
filtered_trips_new.createOrReplaceTempView("trips")

In [0]:
%sql
select * from filtered_trips_new where pickup_zip = 10028 and trip_distance > 10
